# Protein Trajectory Analysis Using Delta Matrix
## UK Biobank (UKB) Cohort

**Author:** Ximing Ran


## Overview

This notebook fits **GAM trajectory models** on delta protein values to identify proteins
with significant expression trajectories relative to ALS diagnosis time (`YrSinceDi`).

**Four analysis subsets:**
- **Affected** (n≈22): Cross-sectional, `k=5` for GAM
- **Convert** (n≈264): Larger group, `k=10` for GAM
- **Combined_short** (Affected + Convert, filtered to `YrSinceDi < 5`): `k=10`
- **Combined** (Affected + Convert, **no** `YrSinceDi` filter): `k=10`

No random effect is used (UKB is cross-sectional: one visit per individual).


## 1. Load Libraries

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(mgcv)
library(patchwork)
library(pheatmap)
library(RColorBrewer)
library(ggplotify)
library(cowplot)
library(scales)
library(tibble)
library(knitr)
library(here)

set.seed(42)
theme_set(
  theme_bw() +
  theme(plot.title  = element_text(face = "bold", size = 14),
        axis.title  = element_text(size = 12),
        axis.text   = element_text(size = 10))
)


## 2. Load Data

In [ ]:
# Delta matrix (wide format: eid x proteins)
delta_matrix_wide <- read.csv(here::here("data", "analysis_data", "ukb",
                                          "delta_matrix", "delta_matrix_wide.csv"))

# Visit info
visit_info <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info",
                                   "visit_info.csv"), row.names = 1)

visit_info <- visit_info %>%
  mutate(
    Group     = factor(Group, levels = c("Healthy control", "Pre-symptomatic", "Phenoconverter", "Pre-hospital", "Clinically manifest ALS")),
    GenoGroup = factor(GenoGroup),
    Sex       = factor(Sex, levels = c("Female", "Male"))
  )

cat("Delta matrix dimensions:", dim(delta_matrix_wide), "\n")
cat("\n=== Group Counts ===\n")
print(table(visit_info$Group))


### 2.1 Load Protein List

In [ ]:

# DE results from Analysis 1 (Miami)
protein_de <- read.csv("../../01-Differentially_Expressed_Proteins/Results/Mix_effect_model_lmer.csv")
all_proteins <- setdiff(colnames(delta_matrix_wide ), "eid")

protein_sign <- protein_de %>%
  filter(significant == "Significant") %>%
  pull(Protein)

# Keep only proteins present in UKB data
protein_sign <- intersect(protein_sign, all_proteins)

cat("Total proteins in UKB data:       ", length(all_proteins), "\n")
cat("Significant DE proteins (Miami):  ", length(protein_sign), "\n")
cat("Overlapping proteins to analyse:  ", length(protein_sign), "\n")


## 3. Define Analysis Subsets

- **Subset 1 (Affected):** n≈22, `k=5` (small sample)
- **Subset 2 (Convert):** n≈264, `k=10`
- **Subset 3 (Combined_short):** Affected + Convert filtered to `YrSinceDi < 5`, `k=10`
- **Subset 4 (Combined):** Affected + Convert with **no** `YrSinceDi` filter, `k=10`


In [ ]:
visit_affected <- visit_info %>%
  filter(Group == "Clinically manifest ALS", !is.na(YrSinceDi))

visit_convert <- visit_info %>%
  filter(Group %in% c("Phenoconverter","Pre-hospital"), !is.na(YrSinceDi))

visit_combined_short <- visit_info %>%
  filter(Group %in% c("Clinically manifest ALS", "Phenoconverter","Pre-hospital"), !is.na(YrSinceDi)) %>%
  filter(YrSinceDi < 5)

visit_combined <- visit_info %>%
  filter(Group %in% c("Clinically manifest ALS", "Phenoconverter","Pre-hospital"), !is.na(YrSinceDi))

cat("Subset 1 - Affected:        ", nrow(visit_affected),
    "| YrSinceDi range:",
    round(min(visit_affected$YrSinceDi), 2), "to",
    round(max(visit_affected$YrSinceDi), 2), "\n")

cat("Subset 2 - Convert:         ", nrow(visit_convert),
    "| YrSinceDi range:",
    round(min(visit_convert$YrSinceDi), 2), "to",
    round(max(visit_convert$YrSinceDi), 2), "\n")

cat("Subset 3 - Combined_short:  ", nrow(visit_combined_short),
    "| YrSinceDi range:",
    round(min(visit_combined_short$YrSinceDi), 2), "to",
    round(max(visit_combined_short$YrSinceDi), 2), "\n")

cat("Subset 4 - Combined:        ", nrow(visit_combined),
    "| YrSinceDi range:",
    round(min(visit_combined$YrSinceDi), 2), "to",
    round(max(visit_combined$YrSinceDi), 2), "\n")


## 4. Helper Functions

In [ ]:
get_smooth_pvalue <- function(gam_obj, term = "s(YrSinceDi)") {
  if (is.null(gam_obj)) return(NA_real_)
  tryCatch({
    st <- summary(gam_obj)$s.table
    if (!is.null(st) && term %in% rownames(st)) return(st[term, "p-value"])
    NA_real_
  }, error = function(e) NA_real_)
}

get_significance_start_time <- function(predictions_df) {
  sig_points <- predictions_df %>% filter(sig != 0)
  if (nrow(sig_points) == 0)
    return(list(start_time_increase = NA, start_time_decrease = NA, start_time_any = NA))
  inc <- predictions_df %>% filter(sig ==  1)
  dec <- predictions_df %>% filter(sig == -1)
  list(
    start_time_increase = if (nrow(inc) > 0) min(inc$YrSinceDi) else NA,
    start_time_decrease = if (nrow(dec) > 0) min(dec$YrSinceDi) else NA,
    start_time_any      = min(sig_points$YrSinceDi)
  )
}


## 5. Trajectory Analysis Function

`k` is passed as a parameter so Affected (k=5) and Convert (k=10) can be handled separately.


In [ ]:
fit_delta_trajectory <- function(delta_df, protein_name, k = 10, min_obs = 5) {

  df <- delta_df %>% filter(!is.na(delta), !is.na(YrSinceDi))
  if (nrow(df) < min_obs) {
    message("  Skipping ", protein_name, " (n=", nrow(df), ")")
    return(NULL)
  }

  # Clamp k to at most n-1 to avoid over-parameterisation
  k_use <- min(k, nrow(df) - 1)

  mod <- tryCatch(
    suppressWarnings(
      mgcv::gam(delta ~ s(YrSinceDi, k = k_use), data = df, method = "REML")
    ),
    error = function(e) {
      message("  GAM failed for ", protein_name, ": ", e$message)
      NULL
    }
  )
  if (is.null(mod)) return(NULL)

  pval <- get_smooth_pvalue(mod)

  yrs <- seq(min(df$YrSinceDi, na.rm = TRUE),
             max(df$YrSinceDi, na.rm = TRUE), length.out = 300)
  gp  <- data.frame(YrSinceDi = yrs)
  pr  <- suppressWarnings(predict(mod, newdata = gp, se.fit = TRUE))
  gp$fit   <- as.numeric(pr$fit)
  gp$se    <- as.numeric(pr$se.fit)
  gp$lower <- gp$fit - 1.96 * gp$se
  gp$upper <- gp$fit + 1.96 * gp$se
  gp <- gp %>%
    mutate(sig = case_when(lower > 0 ~  1L, upper < 0 ~ -1L, TRUE ~ 0L),
           dir = case_when(sig > 0 ~ "increase", sig < 0 ~ "decrease",
                           TRUE ~ NA_character_))

  st <- get_significance_start_time(gp)

  list(protein             = protein_name,
       model               = mod,
       data                = df,
       predictions         = gp,
       n_obs               = nrow(df),
       n_subjects          = length(unique(df$eid)),
       pvalue_YrSinceDi    = pval,
       start_time_increase = st$start_time_increase,
       start_time_decrease = st$start_time_decrease,
       start_time_any      = st$start_time_any)
}


## 6. Trajectory Plot Function

In [ ]:
create_trajectory_plot <- function(result, subset_label = "") {
  if (is.null(result)) return(NULL)

  df      <- result$data
  grid    <- result$predictions
  protein <- result$protein
  y_max <- 4; y_min <- -2
  fill_map <- c(increase = "#D73027", decrease = "#4575B4")


  # ─────────────────────────────────────────────────────────────────────────

  sig_points <- grid %>% filter(!is.na(dir))
  r      <- rle(grid$sig)
  ends   <- cumsum(r$lengths)
  starts <- c(1, head(ends, -1) + 1)
  nzi    <- which(r$values != 0)

  if (length(nzi) > 0) {
    run_df <- tibble(
      start_year = grid$YrSinceDi[starts[nzi]],
      end_year   = grid$YrSinceDi[ends[nzi]],
      run_sign   = r$values[nzi]
    ) %>% mutate(
      dir             = ifelse(run_sign > 0, "increase", "decrease"),
      color_hex       = ifelse(run_sign > 0, "#D73027", "#4575B4"),
      y_label_val_ind = ifelse(dir == "increase", -0.4, 0.2)
    )
  } else {
    run_df <- tibble(start_year = numeric(), end_year = numeric(),
                     run_sign = integer(), dir = character(),
                     color_hex = character(), y_label_val_ind = numeric())
  }

  p <- ggplot() +
    geom_point(data = df, aes(x = YrSinceDi, y = delta),
               alpha = 0.6, size = 1.2, color = "grey10") +
    geom_ribbon(data = grid, aes(x = YrSinceDi, ymin = lower, ymax = upper),
                fill = "grey30", alpha = 0.2) +
    geom_line(data = grid, aes(x = YrSinceDi, y = fit),
              color = "black", linewidth = 1.4) +
    geom_hline(yintercept = 0, linetype = "dashed", color = "grey40") +
    geom_vline(xintercept = 0, color = "black") +
    coord_cartesian(ylim = c(y_min, y_max)) +
    labs(
      title    = paste0(protein, if (subset_label != "") paste0(" [", subset_label, "]") else ""),
      subtitle = paste0("N=", result$n_subjects,
                        " | p=", format(result$pvalue_YrSinceDi, digits = 3, scientific = TRUE)),
      x = "Years Since Diagnosis (YrSinceDi)",
      y = "Delta (Observed - Expected)"
    ) +
    theme_bw() +
    theme(plot.title    = element_text(size = 11, face = "bold", hjust = 0.5),
          plot.subtitle = element_text(size = 9,  hjust = 0.5),
          legend.position = "none")

  if (nrow(sig_points) > 0)
    p <- p +
      geom_point(data = sig_points, aes(x = YrSinceDi, y = 0, color = dir),
                 size = 1.2, alpha = 0.9, show.legend = FALSE) +
      scale_color_manual(values = fill_map)

  if (nrow(run_df) > 0)
    p <- p +
      geom_text(data = run_df,
                aes(x = start_year, y = y_label_val_ind, label = round(start_year, 1)),
                color = run_df$color_hex, size = 3.5, fontface = "bold", vjust = 0) +
      geom_text(data = run_df,
                aes(x = end_year, y = y_label_val_ind, label = round(end_year, 1)),
                color = run_df$color_hex, size = 3.5, fontface = "bold", vjust = 0)
  return(p)
}

## 7. Batch Analysis Function

In [ ]:
run_batch_analysis <- function(visit_subset, subset_name, k = 10, outdir_base) {

  subdir <- file.path(outdir_base, gsub(" ", "_", subset_name))
  dir.create(subdir, recursive = TRUE, showWarnings = FALSE)

  results_list <- list()
  plots_list   <- list()

  cat("\n=== Running analysis:", subset_name, "===\n")
  cat("Proteins:", length(protein_sign), "| Samples:", nrow(visit_subset),
      "| k =", k, "\n")

  for (i in seq_along(protein_sign)) {

    protein <- protein_sign[i]

    if (i %% 10 == 0 || i == length(protein_sign))
      cat(sprintf("  Processing %d / %d\n", i, length(protein_sign)))

    delta_df <- delta_matrix_wide %>%
      select(eid, all_of(protein)) %>%
      rename(delta = all_of(protein)) %>%
      inner_join(visit_subset %>% select(eid, YrSinceDi, Sex, GenoGroup), by = "eid")

    result <- fit_delta_trajectory(delta_df, protein, k = k)

    if (!is.null(result)) {
      results_list[[protein]] <- result
      plots_list[[protein]]   <- create_trajectory_plot(result, subset_name)
    }
  }

  # ── Save combined confirm_df and grid_overall for all proteins ────────────
  fig_data_dir <- "./Results/2.Protein_Trajectories_All/Fig_data"
  dir.create(fig_data_dir, recursive = TRUE, showWarnings = FALSE)

  safe_name <- gsub(" ", "_", subset_name)

  confirm_df_all <- bind_rows(lapply(results_list, function(x)
    x$data %>% mutate(protein = x$protein)))

  grid_overall_all <- bind_rows(lapply(results_list, function(x)
    x$predictions %>% mutate(protein = x$protein)))

  write.csv(confirm_df_all,
            file.path(fig_data_dir, paste0("confirm_df_", safe_name, ".csv")),
            row.names = FALSE)
  write.csv(grid_overall_all,
            file.path(fig_data_dir, paste0("grid_overall_", safe_name, ".csv")),
            row.names = FALSE)

  cat("Fig data saved to:", fig_data_dir, "\n")
  # ─────────────────────────────────────────────────────────────────────────

  # Summary table
  summary_df <- tibble(
    protein             = names(results_list),
    n_obs               = sapply(results_list, function(x) x$n_obs),
    n_subjects          = sapply(results_list, function(x) x$n_subjects),
    pvalue_YrSinceDi    = sapply(results_list, function(x) x$pvalue_YrSinceDi),
    start_time_increase = sapply(results_list, function(x) x$start_time_increase),
    start_time_decrease = sapply(results_list, function(x) x$start_time_decrease),
    start_time_any      = sapply(results_list, function(x) x$start_time_any)
  ) %>%
    mutate(
      fdr_YrSinceDi       = p.adjust(pvalue_YrSinceDi, method = "BH"),
      significant_nominal = pvalue_YrSinceDi < 0.05,
      significant_fdr     = fdr_YrSinceDi    < 0.05
    ) %>%
    arrange(pvalue_YrSinceDi)

  write.csv(summary_df,
            file.path(subdir, paste0("summary_", gsub(" ", "_", subset_name), ".csv")),
            row.names = FALSE)

  cat("Models fitted:", length(results_list), "\n")
  cat("Significant (nominal):", sum(summary_df$significant_nominal, na.rm = TRUE), "\n")
  cat("Significant (FDR):    ", sum(summary_df$significant_fdr,     na.rm = TRUE), "\n")

  list(results     = results_list,
       plots       = plots_list,
       summary     = summary_df,
       subset_name = subset_name,
       subdir      = subdir)
}

## 8. Run Analysis for All Four Subsets

In [ ]:
outdir <- file.path("Results","2.Protein_Trajectories_All")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)


### 8.1 Subset 1: Affected (k=5)

In [ ]:
analysis_affected <- run_batch_analysis(
  visit_subset = visit_affected,
  subset_name  = "Affected",
  k            = 5,
  outdir_base  = outdir
)


### 8.2 Subset 2: Convert (k=10)

In [ ]:
analysis_convert <- run_batch_analysis(
  visit_subset = visit_convert,
  subset_name  = "Convert",
  k            = 10,
  outdir_base  = outdir
)


### 8.3 Subset 3: Combined_short (Affected + Convert, YrSinceDi < 5) (k=10)

In [ ]:
analysis_combined_short <- run_batch_analysis(
  visit_subset = visit_combined_short,
  subset_name  = "Combined_short",
  k            = 10,
  outdir_base  = outdir
)


### 8.4 Subset 4: Combined (Affected + Convert, no YrSinceDi filter) (k=10)

In [ ]:
analysis_combined <- run_batch_analysis(
  visit_subset = visit_combined,
  subset_name  = "Combined",
  k            = 10,
  outdir_base  = outdir
)


## 9. Summary Tables

In [ ]:
print_summary_table <- function(analysis) {
  tbl <- analysis$summary %>%
    mutate(
      Index               = row_number(),
      start_time_increase = ifelse(is.na(start_time_increase), "",
                                   round(start_time_increase, 2)),
      start_time_decrease = ifelse(is.na(start_time_decrease), "",
                                   round(start_time_decrease, 2))
    ) %>%
    select(Index, Protein = protein, N = n_subjects,
           Pval = pvalue_YrSinceDi, padj = fdr_YrSinceDi,
           Start_UP = start_time_increase, Start_DOWN = start_time_decrease) %>%
    mutate(across(c(Pval, padj), ~ round(.x, 4)))

  knitr::kable(tbl,
    caption = paste0(analysis$subset_name, ": Proteins Ranked by Trajectory Association"))
}


In [ ]:
cat("=== Affected ===\n")
print_summary_table(analysis_affected)


In [ ]:
cat("=== Convert ===\n")
print_summary_table(analysis_convert)


In [ ]:
cat("=== Combined ===\n")
print_summary_table(analysis_combined)

## 10. P-value Distributions

In [ ]:
options(repr.plot.width = 22, repr.plot.height = 4)

pval_data <- bind_rows(
  analysis_affected$summary %>%
    select(protein, pvalue_YrSinceDi) %>% mutate(subset = "Affected"),
  analysis_convert$summary %>%
    select(protein, pvalue_YrSinceDi) %>% mutate(subset = "Convert"),
  analysis_combined_short$summary %>%
    select(protein, pvalue_YrSinceDi) %>% mutate(subset = "Combined_short"),
  analysis_combined$summary %>%
    select(protein, pvalue_YrSinceDi) %>% mutate(subset = "Combined")
)

pval_data$subset <- factor(pval_data$subset,
  levels = c("Affected", "Convert", "Combined_short", "Combined"))

ggplot(pval_data, aes(x = pvalue_YrSinceDi)) +
  geom_histogram(bins = 40, fill = "steelblue", color = "white", alpha = 0.8) +
  geom_vline(xintercept = 0.05, linetype = "dashed", color = "red") +
  facet_wrap(~subset, ncol = 4, scales = "free_y") +
  labs(title = "P-value Distribution: s(YrSinceDi)", x = "P-value", y = "Count") +
  theme_bw()


## 11. Example Trajectory Plots

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 12)
# Top 6 most significant from each subset
get_top_plots <- function(analysis, n = 6) {
  top <- analysis$summary %>% arrange(pvalue_YrSinceDi) %>%
    head(n) %>% pull(protein)
  top <- intersect(top, names(analysis$plots))
  if (length(top) > 0) {
    print(wrap_plots(analysis$plots[top], ncol = 3) +
            plot_annotation(title = paste("Top", n, "-", analysis$subset_name)))
  }
}
get_top_plots(analysis_affected, n = 6)


In [ ]:
get_top_plots(analysis_convert, n = 6)


In [ ]:
get_top_plots(analysis_combined_short, n = 6)


In [ ]:
get_top_plots(analysis_combined, n = 6)


## 12. Export PDFs

In [ ]:
export_pdfs <- function(analysis) {

  # All proteins
  pdf_path <- file.path(analysis$subdir,
                        paste0("trajectory_plots_all_", analysis$subset_name, ".pdf"))
  pdf(pdf_path, width = 8, height = 6)
  for (p in names(analysis$plots)) {
    if (!is.null(analysis$plots[[p]])) print(analysis$plots[[p]])
  }
  dev.off()
  cat("All-protein PDF saved:", pdf_path, "\n")

  # Significant only
  sp <- analysis$summary %>% filter(significant_fdr) %>% pull(protein)
  sp <- intersect(sp, names(analysis$plots))
  if (length(sp) > 0) {
    ncols <- min(5, length(sp))
    nrows <- ceiling(length(sp) / ncols)
    ggsave(wrap_plots(analysis$plots[sp], ncol = ncols),
           filename = file.path(analysis$subdir,
                                paste0("trajectory_significant_", analysis$subset_name, ".pdf")),
           width = ncols * 4, height = nrows * 4)
    ggsave(wrap_plots(analysis$plots[sp], ncol = ncols),
           filename = file.path(analysis$subdir,
                                paste0("trajectory_significant_", analysis$subset_name, ".png")),
           width = ncols * 4, height = nrows * 4, dpi = 300)
    cat("Significant plots saved (n =", length(sp), ")\n")
  } else {
    cat("No FDR-significant proteins for", analysis$subset_name, "\n")
  }
}
export_pdfs(analysis_affected)
export_pdfs(analysis_convert)
export_pdfs(analysis_combined_short)
export_pdfs(analysis_combined)


## 13. 1-Year Bin Heatmap

In [ ]:
compute_1yr_bin_heatmap_ukb <- function(analysis, bin_width = 1.0) {

  results_list <- analysis$results
  summary_df   <- analysis$summary
  outdir_heat  <- file.path(analysis$subdir, "1yr_bin_heatmap")
  dir.create(outdir_heat, recursive = TRUE, showWarnings = FALSE)

  sig_proteins <- summary_df %>% filter(significant_fdr) %>% pull(protein)

  all_preds <- bind_rows(lapply(names(results_list), function(p)
    results_list[[p]]$predictions %>% mutate(protein = p)))

  if (nrow(all_preds) == 0) { cat("No predictions to heatmap.\n"); return(NULL) }

  min_y       <- floor(min(all_preds$YrSinceDi,   na.rm = TRUE))
  max_y       <- ceiling(max(all_preds$YrSinceDi, na.rm = TRUE))
  bin_breaks  <- seq(min_y, max_y + bin_width - 1, by = bin_width)
  bin_centers <- (head(bin_breaks, -1) + tail(bin_breaks, -1)) / 2

  pbl <- list(); ral <- list()

  for (prot in names(results_list)) {
    grid     <- results_list[[prot]]$predictions
    grid$bin <- cut(grid$YrSinceDi, breaks = bin_breaks,
                    include.lowest = TRUE, right = FALSE)
    agg <- grid %>% filter(!is.na(dir)) %>% group_by(bin) %>%
      summarise(mean_fit = mean(fit, na.rm = TRUE), n_sig_points = n(),
                direction = unique(dir)[1], .groups = "drop")
    ab  <- tibble(
      bin        = levels(cut(bin_centers, breaks = bin_breaks,
                              include.lowest = TRUE, right = FALSE)),
      bin_center = bin_centers
    )
    pbl[[prot]] <- ab %>% left_join(agg, by = "bin") %>%
      mutate(protein = prot) %>%
      select(protein, bin_center, mean_fit, n_sig_points, direction)

    sc  <- case_when(grid$sig ==  1L ~  1L, grid$sig == -1L ~ -1L, TRUE ~ 0L)
    if (all(sc == 0L)) { ral[[prot]] <- tibble(); next }
    r      <- rle(sc); ends <- cumsum(r$lengths); starts <- c(1, head(ends,-1)+1)
    nzi    <- which(r$values != 0)
    if (length(nzi) == 0) { ral[[prot]] <- tibble(); next }
    ral[[prot]] <- tibble(
      protein    = prot,
      start_year = grid$YrSinceDi[starts[nzi]],
      end_year   = grid$YrSinceDi[ends[nzi]],
      run_sign   = r$values[nzi]
    ) %>% mutate(dir       = ifelse(run_sign > 0, "increase", "decrease"),
                 color_hex = ifelse(run_sign > 0, "#D73027", "#4575B4"))
  }

  heat_long   <- bind_rows(pbl)
  runs_all_df <- bind_rows(ral)

  write.csv(heat_long,   file.path(outdir_heat, "heat_1yr_long.csv"),   row.names = FALSE)
  write.csv(runs_all_df, file.path(outdir_heat, "runs_all_1yr.csv"),     row.names = FALSE)

  # Protein ordering
  if (nrow(runs_all_df) == 0 || length(sig_proteins) == 0) {
    final_order <- intersect(unique(heat_long$protein), sig_proteins)
  } else {
    ea  <- runs_all_df %>% group_by(protein) %>%
      summarise(ea = min(start_year, na.rm = TRUE), .groups = "drop")
    final_order <- tibble(protein = intersect(unique(heat_long$protein), sig_proteins)) %>%
      left_join(ea, by = "protein") %>% arrange(ea) %>% pull(protein)
  }

  gc  <- c("blue", "white", "red")
  xb  <- round(min(heat_long$bin_center)):round(max(heat_long$bin_center))

  bh  <- function(pdf, ts = "") {
    mnv <- min(pdf$mean_fit, na.rm = TRUE); mxv <- max(pdf$mean_fit, na.rm = TRUE)
    if (!is.finite(mnv)) mnv <- -1; if (!is.finite(mxv)) mxv <- 1
    ggplot(pdf, aes(x = bin_center, y = protein, fill = mean_fit)) +
      geom_tile(color = "black", linewidth = 0.4) +
      geom_vline(xintercept = 0, color = "black", linewidth = 2) +
      scale_fill_gradientn(colors = gc, values = rescale(c(mnv, 0, mxv)),
                           na.value = "grey", name = "Avg Delta") +
      scale_y_discrete(limits = rev) +
      scale_x_continuous(breaks = xb) +
      labs(x = "YrSinceDi (1yr bin)", y = "",
           title = paste0(analysis$subset_name, " — 1yr bin heatmap", ts)) +
      theme_minimal() +
      theme(axis.text.y = element_text(size = 7), axis.text.x = element_text(size = 7),
            panel.grid = element_blank())
  }

  hp  <- bh(heat_long %>% mutate(protein = factor(protein, levels = unique(protein))))
  ggsave(file.path(outdir_heat, "heatmap_1yr.png"), hp,
         width  = max(10, length(unique(heat_long$bin_center)) * 0.3),
         height = max(6,  length(unique(heat_long$protein))   * 0.12), dpi = 300)

  hps <- NULL
  if (length(final_order) > 0) {
    d   <- heat_long %>% filter(protein %in% final_order) %>%
      mutate(protein = factor(protein, levels = final_order))
    hps <- bh(d, " [sorted, FDR sig]")
    ggsave(file.path(outdir_heat, "heatmap_1yr_sorted.png"), hps,
           width  = max(10, length(unique(d$bin_center)) * 0.3),
           height = max(6,  length(final_order) * 0.15), dpi = 300)
  }

  cat("Heatmap saved to:", outdir_heat, "\n")
  list(heat_long = heat_long, runs_all = runs_all_df, final_order = final_order,
       plot_unsorted = hp, plot_sorted = hps, outdir = outdir_heat)
}


In [ ]:
heatmap_affected       <- compute_1yr_bin_heatmap_ukb(analysis_affected)
heatmap_convert        <- compute_1yr_bin_heatmap_ukb(analysis_convert)
heatmap_combined_short <- compute_1yr_bin_heatmap_ukb(analysis_combined_short)
heatmap_combined       <- compute_1yr_bin_heatmap_ukb(analysis_combined)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 8)

if (!is.null(heatmap_affected$plot_sorted)) {
  cat("Affected FDR-sig proteins:", length(heatmap_affected$final_order), "\n")
  print(heatmap_affected$plot_sorted)
}


In [ ]:
if (!is.null(heatmap_convert$plot_sorted)) {
  cat("Convert FDR-sig proteins:", length(heatmap_convert$final_order), "\n")
  print(heatmap_convert$plot_sorted)
}


## 14. Cross-Subset Comparison

In [ ]:
sig_affected       <- analysis_affected$summary       %>% filter(significant_fdr) %>% pull(protein)
sig_convert        <- analysis_convert$summary        %>% filter(significant_fdr) %>% pull(protein)
sig_combined_short <- analysis_combined_short$summary %>% filter(significant_fdr) %>% pull(protein)
sig_combined       <- analysis_combined$summary       %>% filter(significant_fdr) %>% pull(protein)

cat("FDR-significant proteins\n")
cat("  Affected:       ", length(sig_affected),       "\n")
cat("  Convert:        ", length(sig_convert),        "\n")
cat("  Combined_short: ", length(sig_combined_short), "\n")
cat("  Combined:       ", length(sig_combined),       "\n")
cat("  All 4:          ",
    length(Reduce(intersect,
                  list(sig_affected, sig_convert, sig_combined_short, sig_combined))), "\n")

combined_summary <- Reduce(
  function(a, b) full_join(a, b, by = "protein"),
  list(
    analysis_affected$summary %>%
      select(protein, pvalue_affected = pvalue_YrSinceDi,
             fdr_affected = fdr_YrSinceDi, sig_affected = significant_fdr,
             start_any_affected = start_time_any),
    analysis_convert$summary %>%
      select(protein, pvalue_convert = pvalue_YrSinceDi,
             fdr_convert = fdr_YrSinceDi, sig_convert = significant_fdr,
             start_any_convert = start_time_any),
    analysis_combined_short$summary %>%
      select(protein, pvalue_combined_short = pvalue_YrSinceDi,
             fdr_combined_short = fdr_YrSinceDi, sig_combined_short = significant_fdr,
             start_any_combined_short = start_time_any),
    analysis_combined$summary %>%
      select(protein, pvalue_combined = pvalue_YrSinceDi,
             fdr_combined = fdr_YrSinceDi, sig_combined = significant_fdr,
             start_any_combined = start_time_any)
  )
)

write.csv(combined_summary,
          file.path(outdir, "combined_summary_all_subsets.csv"),
          row.names = FALSE)

cat("Combined summary saved.\n")


In [ ]:
# One row per protein, 8 columns (padj + start time for each of 4 subsets)
format_padj <- function(val) {
  if (is.na(val)) return("")
  if (val < 0.001) return("<0.001")
  formatC(val, digits = 3, format = "f")
}

format_start <- function(padj, start) {
  if (is.na(padj) || padj >= 0.05 || is.na(start)) return("")
  formatC(start, digits = 1, format = "f")
}

summary_table <- Reduce(
  function(a, b) full_join(a, b, by = "protein"),
  list(
    analysis_affected$summary %>%
      select(protein, fdr_affected = fdr_YrSinceDi, start_affected = start_time_any),
    analysis_convert$summary %>%
      select(protein, fdr_convert = fdr_YrSinceDi, start_convert = start_time_any),
    analysis_combined_short$summary %>%
      select(protein, fdr_combined_short = fdr_YrSinceDi, start_combined_short = start_time_any),
    analysis_combined$summary %>%
      select(protein, fdr_combined = fdr_YrSinceDi, start_combined = start_time_any)
  )
) %>%
rowwise() %>%
mutate(
  `padj (Affected)`        = format_padj(fdr_affected),
  `Start (Affected)`       = format_start(fdr_affected,       start_affected),
  `padj (Convert)`         = format_padj(fdr_convert),
  `Start (Convert)`        = format_start(fdr_convert,        start_convert),
  `padj (Combined_short)`  = format_padj(fdr_combined_short),
  `Start (Combined_short)` = format_start(fdr_combined_short, start_combined_short),
  `padj (Combined)`        = format_padj(fdr_combined),
  `Start (Combined)`       = format_start(fdr_combined,       start_combined)
) %>%
ungroup() %>%
select(Protein = protein,
       `padj (Affected)`,       `Start (Affected)`,
       `padj (Convert)`,        `Start (Convert)`,
       `padj (Combined_short)`, `Start (Combined_short)`,
       `padj (Combined)`,       `Start (Combined)`) %>%
arrange(Protein)

write.csv(summary_table,
          file.path(outdir, "trajectory_summary_table.csv"),
          row.names = FALSE)

knitr::kable(summary_table,
             caption = "Trajectory Analysis Summary: padj and Earliest Significant Change Time",
             align = c("l", rep("r", 8)))


## 15. Output Files

| File | Description |
|------|-------------|
| `Affected/summary_Affected.csv` | GAM results — Affected |
| `Convert/summary_Convert.csv` | GAM results — Convert |
| `Combined_short/summary_Combined_short.csv` | GAM results — Combined_short (YrSinceDi < 2) |
| `Combined/summary_Combined.csv` | GAM results — Combined (no filter) |
| `*/trajectory_plots_all_*.pdf` | All-protein plots per subset |
| `*/trajectory_significant_*.pdf/png` | FDR-significant plots per subset |
| `*/1yr_bin_heatmap/heatmap_1yr_sorted.png` | Sorted 1yr bin heatmap per subset |
| `*/temporal_heatmap_*.png` | Pheatmap per subset |
| `combined_summary_all_subsets.csv` | Side-by-side results for all 4 subsets |
| `trajectory_summary_table.csv` | 8-column padj + start time table |


In [ ]:
# ── Temporal pheatmap using Miami annotation path ─────────────────────────────

annotation_path <- here::here("data", "analysis_data", "miami", "protein_data",
  "Heatmap_PPI_cluster_mask_with_cluster_annotation_row.csv")

if (!file.exists(annotation_path)) {
  cat("Annotation file not found — running without module annotations.\n")
  annotation_path <- NULL
}

build_pheatmap_ukb <- function(heatmap_result, analysis, subset_label = "",
                                annotation_row_path = NULL,
                                key_proteins = c("NEFL", "EDA2R", "MEGF10", "CA14"),
                                adj_p_cutoff = 0.05) {

  if (is.null(heatmap_result) || length(heatmap_result$final_order) == 0) {
    message("No significant proteins for ", subset_label)
    return(NULL)
  }

  # ── Build wide df from heat_long ──────────────────────────────────────────
  heat_long  <- heatmap_result$heat_long
  summary_df <- analysis$summary

  sig_proteins <- summary_df %>% filter(fdr_YrSinceDi < adj_p_cutoff) %>% pull(protein)
  if (length(sig_proteins) == 0) { message("No significant proteins."); return(NULL) }

  heat_sig <- heat_long %>% filter(protein %in% sig_proteins)

  # Wide matrix: protein x bin_center
  hwdf <- heat_sig %>%
    select(protein, bin_center, mean_fit) %>%
    pivot_wider(names_from = bin_center, values_from = mean_fit)

  # Attach p-values and start time
  hwdf <- hwdf %>%
    left_join(summary_df %>% select(protein,
                                     start_year = start_time_any,
                                     Adj_P_value = fdr_YrSinceDi),
              by = "protein")

  df <- hwdf

  # ── Identify time-bin columns ─────────────────────────────────────────────
  meta_cols     <- c("protein", "start_year", "Adj_P_value")
  time_bin_cols <- setdiff(colnames(df), meta_cols)

  bin_numeric <- sapply(time_bin_cols, function(cn) {
    s <- gsub("^X", "", cn)
    if (grepl("^\\.", s)) s <- sub("^\\.", "-", s)
    suppressWarnings(as.numeric(s))
  })
  col_order             <- order(bin_numeric)
  time_bin_cols_ordered <- time_bin_cols[col_order]
  bin_numeric_ordered   <- bin_numeric[col_order]

  # ── Build matrix ──────────────────────────────────────────────────────────
  mat <- df %>%
    select(protein, all_of(time_bin_cols_ordered)) %>%
    column_to_rownames("protein") %>%
    as.matrix()
  mode(mat) <- "numeric"

  colnames(mat) <- ifelse(bin_numeric_ordered < 0,
    paste0("-Y", abs(bin_numeric_ordered)),
    paste0("Y",  bin_numeric_ordered))
  gap_col <- sum(bin_numeric_ordered < 0)

  # ── Impute row mean ───────────────────────────────────────────────────────
  impute_row_mean <- function(x) {
    if (all(is.na(x))) return(rep(0, length(x)))
    x[is.na(x)] <- mean(x, na.rm = TRUE); x
  }
  mat_imputed <- t(apply(mat, 1, impute_row_mean))

  # ── Module annotations ────────────────────────────────────────────────────
  annotation_row_plot <- NULL
  annotation_colors   <- NULL
  gap_indices         <- NULL

  if (!is.null(annotation_row_path) && file.exists(annotation_row_path)) {

    ann_raw <- read.csv(annotation_row_path, row.names = 1) %>%
      select(cluster) %>%
      rownames_to_column("protein")

    module_map <- c(`1` = "ModuleA_Skeletal Muscle",
                    `2` = "ModuleB_TNF Signaling",
                    `3` = "ModuleA_Skeletal Muscle",
                    `5` = "ModuleD_ECM Interaction",
                    `6` = "ModuleD_ECM Interaction",
                    `8` = "ModuleC_Neurofilament")

    module_color_map <- c("ModuleA_Skeletal Muscle" = "#ff0000",
                          "ModuleB_TNF Signaling"   = "#ff8765",
                          "ModuleC_Neurofilament"   = "#aeff65",
                          "ModuleD_ECM Interaction" = "#ffca65",
                          "Others"                  = "#cccccc")

    module_levels <- c("ModuleA_Skeletal Muscle", "ModuleB_TNF Signaling",
                       "ModuleC_Neurofilament",   "ModuleD_ECM Interaction", "Others")

    pm <- tibble(protein = rownames(mat)) %>%
      left_join(ann_raw, by = "protein") %>%
      mutate(
        Module = ifelse(cluster %in% c(1, 2, 3, 5, 6, 8),
                        unname(module_map[as.character(cluster)]), "Others"),
        Module = factor(Module, levels = module_levels, ordered = TRUE)
      ) %>%
      left_join(df %>% select(protein, start_year), by = "protein") %>%
      arrange(Module, start_year)

    mat         <- mat[pm$protein,     , drop = FALSE]
    mat_imputed <- mat_imputed[pm$protein, , drop = FALSE]

    annotation_row_plot <- data.frame(Module = pm$Module)
    rownames(annotation_row_plot) <- pm$protein
    annotation_colors <- list(Module = module_color_map[module_levels])

    mr          <- rle(as.character(pm$Module))
    gap_indices <- cumsum(mr$lengths)
    gap_indices <- gap_indices[gap_indices < nrow(mat)]

    cat("Module annotation applied.\n")

  } else {
    # Sort by earliest significant change time
    sy  <- df %>% select(protein, start_year) %>% deframe()
    ord <- order(sy[rownames(mat)], na.last = TRUE)
    mat         <- mat[ord, , drop = FALSE]
    mat_imputed <- mat_imputed[ord, , drop = FALSE]
    cat("No annotation file — sorted by start_year.\n")
  }

  # ── Mark key proteins ─────────────────────────────────────────────────────
  rn   <- rownames(mat)
  mask <- rn %in% key_proteins
  rownames(mat)[mask]         <- paste0(rn[mask], "**")
  rownames(mat_imputed)[mask] <- paste0(rn[mask], "**")
  if (!is.null(annotation_row_plot)) {
    rna <- rownames(annotation_row_plot)
    rownames(annotation_row_plot)[rna %in% key_proteins] <-
      paste0(rna[rna %in% key_proteins], "**")
  }

  # ── Color scale ───────────────────────────────────────────────────────────
  max_abs <- max(abs(mat_imputed), na.rm = TRUE)
  if (!is.finite(max_abs) || max_abs == 0) max_abs <- 1
  breaks <- seq(-max_abs, max_abs, length.out = 101)
  colors <- colorRampPalette(rev(RColorBrewer::brewer.pal(11, "RdBu")))(100)

  # Number matrix (observed values in tiles)
  nm <- mat
  nm[!is.na(nm)] <- round(as.numeric(nm[!is.na(nm)]), 2)
  nm[is.na(nm)]  <- ""

  # ── Save wide format CSV for heatmap ─────────────────────────────────────
  if (!is.null(heatmap_result$runs_all) && nrow(heatmap_result$runs_all) > 0) {
    dir_df <- heatmap_result$runs_all %>%
      group_by(protein) %>%
      slice_min(start_year, with_ties = FALSE) %>%
      ungroup() %>%
      select(protein, dir)
  } else {
    dir_df <- tibble(protein = character(), dir = character())
  }

  # Build wide matrix with original numeric column names (bin centers)
  wide_df <- df %>%
    select(protein, all_of(time_bin_cols_ordered)) %>%
    rename_with(~ as.character(bin_numeric_ordered), all_of(time_bin_cols_ordered))

  # Add start_year, dir, cluster label/color, P_value, Adj_P_value
  wide_df <- wide_df %>%
    left_join(df %>% select(protein, start_year), by = "protein") %>%
    left_join(dir_df, by = "protein")

  if (!is.null(annotation_row_path) && file.exists(annotation_row_path)) {
    ann_for_csv <- read.csv(annotation_row_path, row.names = 1) %>%
      select(cluster) %>%
      rownames_to_column("protein") %>%
      mutate(
        label = ifelse(cluster %in% c(1,2,3,5,6,8), as.character(cluster), "Unclustered"),
        cluster_color = case_when(
          cluster == 1 ~ "#ff0000",
          cluster == 2 ~ "#ff8765",
          cluster == 3 ~ "#ff0000",
          cluster == 5 ~ "#ffca65",
          cluster == 6 ~ "#ffca65",
          cluster == 8 ~ "#aeff65",
          TRUE          ~ "#D3D3D3"
        )
      ) %>%
      select(protein, label, cluster_color)
    wide_df <- wide_df %>% left_join(ann_for_csv, by = "protein")
  } else {
    wide_df <- wide_df %>% mutate(label = "Unclustered", cluster_color = "#D3D3D3")
  }

  wide_df <- wide_df %>%
    left_join(
      summary_df %>% select(protein,
                             P_value     = pvalue_YrSinceDi,
                             Adj_P_value = fdr_YrSinceDi),
      by = "protein"
    )

  wide_csv_path <- file.path(analysis$subdir, "1yr_bin_heatmap",
                              paste0("heatmap_wide_", gsub(" ", "_", subset_label), ".csv"))
  write.csv(wide_df, wide_csv_path, row.names = FALSE, na = "NA")
  cat("Wide heatmap CSV saved:", wide_csv_path, "\n")
  # ── End wide CSV export ───────────────────────────────────────────────────

    # ── Build pheatmap ────────────────────────────────────────────────────────
  ph <- pheatmap::pheatmap(
    mat,
    color              = colors,
    breaks             = breaks,
    cluster_rows       = FALSE,
    cluster_cols       = FALSE,
    show_rownames      = TRUE,
    show_colnames      = TRUE,
    annotation_row     = annotation_row_plot,
    annotation_colors  = annotation_colors,
    gaps_row           = gap_indices,
    gaps_col           = gap_col,
    display_numbers    = nm,
    fontsize_number    = 10,
    fontsize_row       = 10,
    fontsize_col       = 10,
    fontsize           = 10,
    angle_col          = 0,
    cellwidth          = 30,
    border_color       = "black",
    number_color       = "black",
    na_col             = "white",
    main               = subset_label,
    annotation_names_row = FALSE
  )

  p <- ggplotify::as.ggplot(ph$gtable) +
    theme(plot.background  = element_rect(fill = "transparent", color = NA),
          panel.background = element_rect(fill = "transparent", color = NA))

  list(pheatmap = ph, ggplot = p)
}


In [ ]:
# ── Run pheatmap for all four subsets ────────────────────────────────────────
options(repr.plot.width = 20, repr.plot.height = 10)

ph_affected <- build_pheatmap_ukb(
  heatmap_result      = heatmap_affected,
  analysis            = analysis_affected,
  subset_label        = "UKB Affected — YrSinceDi",
  annotation_row_path = annotation_path
)
if (!is.null(ph_affected)) {
  print(ph_affected$ggplot)
  ggsave(file.path(analysis_affected$subdir, "temporal_heatmap_affected.png"),
         plot = ph_affected$ggplot, width = 20, height = 8, dpi = 300)
}

ph_convert <- build_pheatmap_ukb(
  heatmap_result      = heatmap_convert,
  analysis            = analysis_convert,
  subset_label        = "UKB Convert — YrSinceDi",
  annotation_row_path = annotation_path
)
if (!is.null(ph_convert)) {
  print(ph_convert$ggplot)
  ggsave(file.path(analysis_convert$subdir, "temporal_heatmap_convert.png"),
         plot = ph_convert$ggplot, width = 20, height = 8, dpi = 300)
}

ph_combined_short <- build_pheatmap_ukb(
  heatmap_result      = heatmap_combined_short,
  analysis            = analysis_combined_short,
  subset_label        = "UKB Combined_short (Affected + Convert, YrSinceDi < 5)",
  annotation_row_path = annotation_path
)
if (!is.null(ph_combined_short)) {
  print(ph_combined_short$ggplot)
  ggsave(file.path(analysis_combined_short$subdir, "temporal_heatmap_combined_short.png"),
         plot = ph_combined_short$ggplot, width = 20, height = 12, dpi = 300)
}

ph_combined <- build_pheatmap_ukb(
  heatmap_result      = heatmap_combined,
  analysis            = analysis_combined,
  subset_label        = "UKB Combined (Affected + Convert, no filter)",
  annotation_row_path = annotation_path
)
if (!is.null(ph_combined)) {
  # print(ph_combined$ggplot)
  ggsave(file.path(analysis_combined$subdir, "temporal_heatmap_combined.png"),
         plot = ph_combined$ggplot, width = 20, height = 12, dpi = 300)
}
